# C3 PP1 — Product × Aspect × Quarter Aggregation

This notebook converts the temporary C1-style input into the core temporal table used by Component 3.

## Input schema
- review_id
- parent_asin
- timestamp
- review_text
- aspect
- sentiment_label
- sentiment_score
- confidence

## Output unit
**Product × Aspect × Quarter**

For each group, the notebook calculates:
- review count
- mean sentiment
- median sentiment
- positive / neutral / negative rates
- average confidence
- confidence-weighted sentiment
- first / last timestamp


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    COMPONENT_ROOT = WORKSPACE.parent
elif (WORKSPACE / "data" / "processed").exists():
    COMPONENT_ROOT = WORKSPACE
else:
    COMPONENT_ROOT = WORKSPACE / "component_3"

INPUT = COMPONENT_ROOT / "data" / "processed" / "dummy_c1_output.csv"
OUTPUT_DIR = COMPONENT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTPUT_DIR / "paq_series.csv"


## 1. Load the temporary C1 output


In [10]:
df = pd.read_csv(INPUT)

print("Rows:", len(df))
print("Products:", df["parent_asin"].nunique())
print("Aspects:", sorted(df["aspect"].dropna().unique()))

display(df.head())


Rows: 4978
Products: 10
Aspects: ['battery', 'camera', 'display', 'keyboard', 'performance', 'thermal']


,review_id,parent_asin,timestamp,review_text,aspect,sentiment_label,sentiment_score,confidence
0,DUMMY_R002422,B00FFJ0HUE,2021-01-03T20:22:40,Performance is acceptable for basic use.,performance,neutral,0.1911,0.7555
1,DUMMY_R002455,B00FFJ0HUE,2021-01-04T21:54:07,Screen quality is sharp and easy to read. I no...,display,positive,0.6695,0.8049
2,DUMMY_R002515,B00FFJ0HUE,2021-01-08T20:54:08,Heat has not been a problem for me.,thermal,positive,0.4124,0.7215
3,DUMMY_R002385,B00FFJ0HUE,2021-01-09T05:39:37,The battery is acceptable but not outstanding....,battery,neutral,-0.1802,0.7241
4,DUMMY_R002456,B00FFJ0HUE,2021-01-13T08:20:50,Screen quality is average.,display,neutral,0.1344,0.7255


## 2. Validate the C1 → C3 input contract


In [11]:
required = [
    "review_id",
    "parent_asin",
    "timestamp",
    "review_text",
    "aspect",
    "sentiment_label",
    "sentiment_score",
    "confidence",
]

missing = [column for column in required if column not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Schema validation passed.")


Schema validation passed.


## 3. Clean and normalize fields


In [12]:
work = df.copy()

work["timestamp"] = pd.to_datetime(
    work["timestamp"],
    utc=True,
    errors="coerce"
)

work["sentiment_score"] = pd.to_numeric(
    work["sentiment_score"],
    errors="coerce"
)

work["confidence"] = pd.to_numeric(
    work["confidence"],
    errors="coerce"
)

work = work.dropna(
    subset=[
        "review_id",
        "parent_asin",
        "timestamp",
        "aspect",
        "sentiment_label",
        "sentiment_score",
        "confidence",
    ]
).copy()

work["aspect"] = (
    work["aspect"]
    .astype(str)
    .str.lower()
    .str.strip()
)

work["sentiment_label"] = (
    work["sentiment_label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

work = work[
    work["sentiment_score"].between(-1, 1)
    & work["confidence"].between(0, 1)
    & work["sentiment_label"].isin(
        ["positive", "neutral", "negative"]
    )
].copy()

work["quarter"] = (
    work["timestamp"]
    .dt.tz_localize(None)
    .dt.to_period("Q")
    .astype(str)
)

print("Rows after cleaning:", len(work))
display(work.head())


Rows after cleaning: 4978


,review_id,parent_asin,timestamp,review_text,aspect,sentiment_label,sentiment_score,confidence,quarter
0,DUMMY_R002422,B00FFJ0HUE,2021-01-03 20:22:40+00:00,Performance is acceptable for basic use.,performance,neutral,0.1911,0.7555,2021Q1
1,DUMMY_R002455,B00FFJ0HUE,2021-01-04 21:54:07+00:00,Screen quality is sharp and easy to read. I no...,display,positive,0.6695,0.8049,2021Q1
2,DUMMY_R002515,B00FFJ0HUE,2021-01-08 20:54:08+00:00,Heat has not been a problem for me.,thermal,positive,0.4124,0.7215,2021Q1
3,DUMMY_R002385,B00FFJ0HUE,2021-01-09 05:39:37+00:00,The battery is acceptable but not outstanding....,battery,neutral,-0.1802,0.7241,2021Q1
4,DUMMY_R002456,B00FFJ0HUE,2021-01-13 08:20:50+00:00,Screen quality is average.,display,neutral,0.1344,0.7255,2021Q1


## 4. Create helper indicators


In [13]:
work["is_positive"] = (
    work["sentiment_label"] == "positive"
).astype(int)

work["is_neutral"] = (
    work["sentiment_label"] == "neutral"
).astype(int)

work["is_negative"] = (
    work["sentiment_label"] == "negative"
).astype(int)

work["weighted_sentiment_component"] = (
    work["sentiment_score"] * work["confidence"]
)


## 5. Build Product × Aspect × Quarter table


In [14]:
paq = (
    work
    .groupby(
        ["parent_asin", "aspect", "quarter"]
    )
    .agg(
        review_count=("review_id", "count"),
        mean_sentiment=("sentiment_score", "mean"),
        median_sentiment=("sentiment_score", "median"),
        positive_rate=("is_positive", "mean"),
        neutral_rate=("is_neutral", "mean"),
        negative_rate=("is_negative", "mean"),
        avg_confidence=("confidence", "mean"),
        weighted_sentiment_sum=(
            "weighted_sentiment_component",
            "sum"
        ),
        confidence_sum=("confidence", "sum"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
    )
    .reset_index()
)

paq["confidence_weighted_sentiment"] = np.where(
    paq["confidence_sum"] > 0,
    paq["weighted_sentiment_sum"]
    / paq["confidence_sum"],
    np.nan
)

paq = paq.drop(
    columns=[
        "weighted_sentiment_sum",
        "confidence_sum"
    ]
)

numeric_columns = [
    "mean_sentiment",
    "median_sentiment",
    "positive_rate",
    "neutral_rate",
    "negative_rate",
    "avg_confidence",
    "confidence_weighted_sentiment",
]

for column in numeric_columns:
    paq[column] = paq[column].round(4)

paq = (
    paq
    .sort_values(
        ["parent_asin", "aspect", "quarter"]
    )
    .reset_index(drop=True)
)

paq.to_csv(
    OUTPUT,
    index=False
)

print("PAQ rows:", len(paq))
print("Saved to:", OUTPUT)

display(paq.head(20))


PAQ rows: 550
Saved to: c:\Users\victus\Desktop\Repository\J26-DS-349\component_3\data\processed\paq_series.csv


,parent_asin,aspect,quarter,review_count,mean_sentiment,median_sentiment,positive_rate,neutral_rate,negative_rate,avg_confidence,first_timestamp,last_timestamp,confidence_weighted_sentiment
0,B00FFJ0HUE,battery,2021Q1,3,0.1326,-0.1452,0.3333,0.6667,0.0,0.7348,2021-01-09 05:39:37+00:00,2021-03-02 01:21:27+00:00,0.1754
1,B00FFJ0HUE,battery,2021Q2,3,-0.0371,-0.0177,0.0000,1.0000,0.0,0.7411,2021-04-21 11:19:50+00:00,2021-05-03 13:49:34+00:00,-0.0449
2,B00FFJ0HUE,battery,2021Q3,5,0.4642,0.4297,0.8000,0.2000,0.0,0.7929,2021-07-01 16:16:52+00:00,2021-09-18 16:39:42+00:00,0.4900
3,B00FFJ0HUE,battery,2021Q4,5,0.2114,0.1513,0.4000,0.6000,0.0,0.7260,2021-10-25 21:56:46+00:00,2021-12-31 04:21:48+00:00,0.2208
4,B00FFJ0HUE,battery,2022Q1,2,0.0833,0.0833,0.5000,0.0000,0.5,0.8004,2022-01-07 02:20:07+00:00,2022-01-30 10:17:47+00:00,0.0905
5,B00FFJ0HUE,battery,2022Q2,6,0.3849,0.3220,0.6667,0.3333,0.0,0.7768,2022-04-02 05:27:16+00:00,2022-06-11 13:28:33+00:00,0.4034
6,B00FFJ0HUE,battery,2022Q3,6,0.6222,0.7684,0.8333,0.1667,0.0,0.8082,2022-07-10 05:43:44+00:00,2022-08-16 16:59:25+00:00,0.6455
7,B00FFJ0HUE,battery,2022Q4,2,0.3040,0.3040,1.0000,0.0000,0.0,0.7103,2022-10-15 06:14:15+00:00,2022-12-24 21:57:57+00:00,0.3057
8,B00FFJ0HUE,battery,2023Q1,2,0.4198,0.4198,1.0000,0.0000,0.0,0.7904,2023-02-02 01:21:35+00:00,2023-03-18 02:53:49+00:00,0.4172
9,B00FFJ0HUE,battery,2023Q2,1,0.3722,0.3722,1.0000,0.0000,0.0,0.7591,2023-06-16 10:17:06+00:00,2023-06-16 10:17:06+00:00,0.3722


## 6. Integrity checks


In [15]:
rate_sum = (
    paq["positive_rate"]
    + paq["neutral_rate"]
    + paq["negative_rate"]
)

assert np.allclose(
    rate_sum,
    1.0,
    atol=0.001
), "Sentiment rates do not sum to 1."

assert (
    paq["review_count"] > 0
).all()

assert (
    paq["mean_sentiment"]
    .between(-1, 1)
    .all()
)

assert (
    paq["avg_confidence"]
    .between(0, 1)
    .all()
)

print("All integrity checks passed.")


All integrity checks passed.


## 7. Inspect one Product × Aspect temporal series


In [16]:
example_product = (
    paq["parent_asin"]
    .iloc[0]
)

example_aspect = (
    paq.loc[
        paq["parent_asin"] == example_product,
        "aspect"
    ]
    .iloc[0]
)

example = paq[
    (paq["parent_asin"] == example_product)
    & (paq["aspect"] == example_aspect)
].copy()

print("Example product:", example_product)
print("Example aspect:", example_aspect)

display(
    example[
        [
            "quarter",
            "review_count",
            "mean_sentiment",
            "negative_rate",
            "avg_confidence",
            "confidence_weighted_sentiment",
        ]
    ]
)


Example product: B00FFJ0HUE
Example aspect: battery


,quarter,review_count,mean_sentiment,negative_rate,avg_confidence,confidence_weighted_sentiment
0,2021Q1,3,0.1326,0.0,0.7348,0.1754
1,2021Q2,3,-0.0371,0.0,0.7411,-0.0449
2,2021Q3,5,0.4642,0.0,0.7929,0.4900
3,2021Q4,5,0.2114,0.0,0.7260,0.2208
4,2022Q1,2,0.0833,0.5,0.8004,0.0905
5,2022Q2,6,0.3849,0.0,0.7768,0.4034
6,2022Q3,6,0.6222,0.0,0.8082,0.6455
7,2022Q4,2,0.3040,0.0,0.7103,0.3057
8,2023Q1,2,0.4198,0.0,0.7904,0.4172
9,2023Q2,1,0.3722,0.0,0.7591,0.3722


## 8. Development-only low-support flag

For the first development pass, fewer than 5 aspect-level records in a quarter is flagged as low support.

This threshold is temporary and must later be selected through EDA and validation.


In [17]:
DEV_LOW_SUPPORT_THRESHOLD = 5

paq["low_support_dev"] = (
    paq["review_count"]
    < DEV_LOW_SUPPORT_THRESHOLD
)

support_summary = (
    paq
    .groupby(
        ["parent_asin", "aspect"]
    )
    .agg(
        quarters=("quarter", "nunique"),
        low_support_quarters=(
            "low_support_dev",
            "sum"
        ),
        min_support=(
            "review_count",
            "min"
        ),
        median_support=(
            "review_count",
            "median"
        ),
        max_support=(
            "review_count",
            "max"
        ),
    )
    .reset_index()
)

support_summary[
    "low_support_share"
] = (
    support_summary[
        "low_support_quarters"
    ]
    / support_summary["quarters"]
).round(3)

display(
    support_summary
    .sort_values(
        "low_support_share",
        ascending=False
    )
    .head(20)
)


,parent_asin,aspect,quarters,low_support_quarters,min_support,median_support,max_support,low_support_share
1,B00FFJ0HUE,display,11,11,1,3.0,4,1.000
2,B00FFJ0HUE,keyboard,11,10,1,2.0,5,0.909
3,B00FFJ0HUE,performance,11,10,1,3.0,6,0.909
5,B00KB7O49K,battery,11,9,1,2.0,6,0.818
8,B00KB7O49K,performance,11,9,1,3.0,6,0.818
4,B00FFJ0HUE,thermal,11,8,2,2.0,6,0.727
7,B00KB7O49K,display,11,8,1,2.0,6,0.727
6,B00KB7O49K,camera,11,8,1,3.0,6,0.727
9,B00KB7O49K,thermal,11,8,1,3.0,6,0.727
0,B00FFJ0HUE,battery,11,7,1,3.0,6,0.636
